# Compound Comparators

## The Problem: Fields That Belong Together

Per-field comparators score each field independently. That works for most fields,
but sometimes fields only make sense as a group.

**Example:** A person has `surname` and `name`. Gold is "Smith, John", extracted is
"Smith, Jane". Independent scoring gives:
- `surname`: "Smith" vs "Smith" -> match (score 1.0)
- `name`: "John" vs "Jane" -> mismatch (score 0.0)

The extractor gets credit for "Smith" -- but "Smith, Jane" is a **completely different
person** from "Smith, John". The surname match is misleading.

## The Solution: Compound Comparators

A compound comparator is a special kind of **batch comparator** (introduced in
`04_example_semantic`). Both receive multiple fields at once instead of scoring
one field at a time, but they serve different purposes:

| | General batch comparator (e.g. `semantic`) | Compound comparator                                                                                                   |
|---|---|-----------------------------------------------------------------------------------------------------------------------|
| **Groups by** | Comparator name -- all fields using `"semantic"` in one record are batched together | **Parent path** -- fields are grouped by their parent object (e.g. `student.surname` + `student.name`)                |
| **Scores** | Each field independently (one score per field) | The **group as a unit** -- one joint decision for all fields in the group                                             |
| **Typical use** | Send multiple fields to an LLM in one API call for efficiency | Fields that are semantically coupled (name parts, value + unit, address components)                                   |
| **Field count in metrics** | All fields count | Only the **primary** field counts; others become `skipped`, so that the compound fields only get one score as a whole |
| **Base class** | Implement the batch protocol directly | Subclass `CompoundComparator` -- handles grouping, primary/skip, incomplete groups                                    |

Key concepts:
- **Fields** -- which sibling field names form the compound (e.g. `["surname", "name"]`)
- **Primary** -- one field gets the score; the others become `skipped` (so they don't
  double-count in metrics)
- **Group by parent** -- when the same fields appear in multiple places (e.g. student
  vs teacher), each parent gets its own group

## Data

Each record has a student and a teacher, each with `surname`, `name`, and `gender`.
The compound comparator only groups `surname` + `name` -- `gender` is scored
independently with `exact`.

In [9]:
GOLD = [
    {
        "student": {"surname": "Smith", "name": "John", "gender": "male"},
        "teacher": {"surname": "Chen", "name": "Wei", "gender": "female"},
    },
    {
        "student": {"surname": "Kim", "name": "Soo", "gender": "female"},
        "teacher": {"surname": "Mueller", "name": "Anna", "gender": "female"},
    },
]

EXTRACTED = [
    {   # student name wrong (and gender changed to match the wrong name)
        "student": {"surname": "Smith", "name": "Jane", "gender": "female"},
        "teacher": {"surname": "Chen", "name": "Wei", "gender": "female"},
    },
    {   # student correct, teacher surname wrong
        "student": {"surname": "Kim", "name": "Soo", "gender": "female"},
        "teacher": {"surname": "Schmidt", "name": "Anna", "gender": "female"},
    },
]

## Method 1: User `exact` Comparator: Independent Scoring (the problem)

Each field scored on its own with `exact`.

In [14]:
from struct_extract_eval import evaluate

PERSON_FIELDS = {
    "surname": {"type": "string", "x-eval-compare": "exact"},
    "name": {"type": "string", "x-eval-compare": "exact"},
    "gender": {"type": "string", "x-eval-compare": "exact"},
}
independent_schema = {
    "type": "object",
    "properties": {
        "student": {"type": "object", "properties": dict(PERSON_FIELDS)},
        "teacher": {"type": "object", "properties": dict(PERSON_FIELDS)},
    },
}

run_independent = evaluate(GOLD, EXTRACTED, schema=independent_schema)

print(f"Independent scoring -- Mean F1: {run_independent.mean_f1:.3f}")
print()
for record in run_independent.records:
    print(f"  Record {record.record_id} (F1: {record.f1:.3f}):")
    for fr in record.field_results:
        print(f"    {fr.path:<25} {fr.score:.1f}  {fr.status}")

Independent scoring -- Mean F1: 0.750

  Record 0 (F1: 0.667):
    student.surname           1.0  match
    student.name              0.0  mismatch
    student.gender            0.0  mismatch
    teacher.surname           1.0  match
    teacher.name              1.0  match
    teacher.gender            1.0  match
  Record 1 (F1: 0.833):
    student.surname           1.0  match
    student.name              1.0  match
    student.gender            1.0  match
    teacher.surname           0.0  mismatch
    teacher.name              1.0  match
    teacher.gender            1.0  match


Record 0: `student.surname`="Smith" matches, giving misleading credit.
But "Smith, Jane" is not "Smith, John" -- the full name is wrong.

## Method 2: Compound Comparator

step 1 : Define a Compound Comparator

Subclass `CompoundComparator` and implement `compare()`. The base class handles
grouping by parent, primary/skip logic, and incomplete groups.

- `fields`: which sibling field names form the compound
- `primary`: which field gets the score (others become `skipped`)
- `compare(gold, extracted)`: receives dicts of `{field_name: value}`, returns a score

In [ ]:
from struct_extract_eval import CompoundComparator


class NameCompoundComparator(CompoundComparator):
    """Score surname + name as one full name."""

    def __init__(self) -> None:
        super().__init__(
            fields=["surname", "name"],
            primary="surname",        # surname gets the score, name becomes skipped
            name="name_compound",
        )

    def compare(self, gold: dict[str, object], extracted: dict[str, object]) -> float:
        """Compare two full names. Case-insensitive."""
        gold_full = f"{gold['name']} {gold['surname']}"
        ext_full = f"{extracted['name']} {extracted['surname']}"
        return 1.0 if gold_full == ext_full else 0.0

Step 2: Register and Evaluate

Register the comparator, then set `surname` and `name` to use it. `gender` stays
as `exact` -- it's not part of the compound group.

In [16]:
from struct_extract_eval.core.comparators.registry import register

register("name_compound", NameCompoundComparator(), overwrite=True)

# surname + name use compound; gender is scored independently
COMPOUND_PERSON_FIELDS = {
    "surname": {"type": "string", "x-eval-compare": "name_compound"},
    "name": {"type": "string", "x-eval-compare": "name_compound"},
    "gender": {"type": "string", "x-eval-compare": "exact"},
}
compound_schema = {
    "type": "object",
    "properties": {
        "student": {"type": "object", "properties": dict(COMPOUND_PERSON_FIELDS)},
        "teacher": {"type": "object", "properties": dict(COMPOUND_PERSON_FIELDS)},
    },
}

run_compound = evaluate(GOLD, EXTRACTED, schema=compound_schema)

print(f"Compound scoring -- Mean F1: {run_compound.mean_f1:.3f}")
print()
for record in run_compound.records:
    print(f"  Record {record.record_id} (F1: {record.f1:.3f}):")
    for fr in record.field_results:
        reason = f"  ({fr.reason})" if fr.reason else ""
        print(f"    {fr.path:<25} {fr.score:.1f}  {fr.status:<12}{reason}")

Compound scoring -- Mean F1: 0.625

  Record 0 (F1: 0.500):
    student.surname           0.0  mismatch      (compound: mismatch)
    student.name              0.0  skipped       (compound with student.surname)
    student.gender            0.0  mismatch      (mismatch)
    teacher.surname           1.0  match         (compound: match)
    teacher.name              0.0  skipped       (compound with teacher.surname)
    teacher.gender            1.0  match       
  Record 1 (F1: 0.750):
    student.surname           1.0  match         (compound: match)
    student.name              0.0  skipped       (compound with student.surname)
    student.gender            1.0  match       
    teacher.surname           0.0  mismatch      (compound: mismatch)
    teacher.name              0.0  skipped       (compound with teacher.surname)
    teacher.gender            1.0  match       


### What Changed

- `student.surname` now scores the **full name** ("John Smith" vs "Jane Smith" -> mismatch).
  No more misleading credit for a matching surname.
- `student.name` is `skipped` -- it's the supporting field. It contributed to the compound
  decision but doesn't count separately in metrics.
- `student.gender` is scored independently with `exact` -- it's not part of the compound.
  Only `surname` and `name` are grouped together.
- Total scored fields: 4 per record (surname + gender per person). Without compound it
  would be 6 (surname + name + gender per person).

### How Grouping Works

The comparator receives the `name_compound` fields per record and groups them by
**parent path**. Fields using other comparators (like `gender` with `exact`) are
not affected:

```
student.surname + student.name  ->  parent "student"  ->  one full name comparison
teacher.surname + teacher.name  ->  parent "teacher"  ->  another full name comparison
student.gender                  ->  scored independently with exact
teacher.gender                  ->  scored independently with exact
```

## Compound Inside Arrays

When the same fields appear inside array elements, instance paths (`students[0]`,
`students[1]`) ensure each element gets its own group.

In [17]:
GOLD_ARRAY = [
    {"students": [
        {"surname": "Smith", "name": "John", "gender": "male"},
        {"surname": "Kim", "name": "Soo", "gender": "female"},
    ]},
]

EXTRACTED_ARRAY = [
    {"students": [
        {"surname": "Smith", "name": "Jane", "gender": "female"},  # wrong name
        {"surname": "Kim", "name": "Soo", "gender": "female"},     # correct
    ]},
]

array_schema = {
    "type": "object",
    "properties": {
        "students": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "surname": {"type": "string", "x-eval-compare": "name_compound"},
                    "name": {"type": "string", "x-eval-compare": "name_compound"},
                    "gender": {"type": "string", "x-eval-compare": "exact"},
                },
            },
        },
    },
}

run_array = evaluate(GOLD_ARRAY, EXTRACTED_ARRAY, schema=array_schema)

print(f"Compound inside array -- Mean F1: {run_array.mean_f1:.3f}")
print()
for fr in run_array.records[0].field_results:
    reason = f"  ({fr.reason})" if fr.reason else ""
    print(f"  {fr.path:<30} {fr.score:.1f}  {fr.status:<12}{reason}")

Compound inside array -- Mean F1: 0.500

  students[0].surname            0.0  mismatch      (compound: mismatch)
  students[0].name               0.0  skipped       (compound with students[0].surname)
  students[0].gender             0.0  mismatch      (mismatch)
  students[1].surname            1.0  match         (compound: match)
  students[1].name               0.0  skipped       (compound with students[1].surname)
  students[1].gender             1.0  match       
